## 1. Setup and Data Loading
Imports libraries and loads the protein dataset.

# Transfomer without Embeddings on the small dataset

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

# Pre-process sequences
df['seq'] = df['seq'].str.replace("*", "X") # Replace non-standard aa
df = df[df['has_nonstd_aa'] == False].reset_index(drop=True)

print(df.head())
df.info()

  pdb_id chain_code                   seq                  sst8  \
0   1FV1          F  NPVVHFFKNIVTPRTPPPSQ  CCCCCBCCCCCCCCCCCCCC   
1   1LM8          H  DLDLEMLAPYIPMDDDFQLR  CCCCCCCCCBCCSCCCEECC   
2   1O06          A  EEDPDLKAAIQESLREAEEA  CCCHHHHHHHHHHHHHHHTC   
3   1RDQ          I  TTYADFIASGRTGRRNAIHD  CHHHHHHTSSCSSCCCCEEC   
4   1T6O          B  QDSRRSADALLRLQAMAGIS  CHHHHHHHHHHHHHHHHTCC   

                   sst3  len  has_nonstd_aa Exptl.  resolution  R-factor  \
0  CCCCCECCCCCCCCCCCCCC   20          False   XRAY        1.90      0.23   
1  CCCCCCCCCECCCCCCEECC   20          False   XRAY        1.85      0.20   
2  CCCHHHHHHHHHHHHHHHCC   20          False   XRAY        1.45      0.19   
3  CHHHHHHCCCCCCCCCCEEC   20          False   XRAY        1.26      0.13   
4  CHHHHHHHHHHHHHHHHCCC   20          False   XRAY        2.00      0.23   

   FreeRvalue  
0        0.27  
1        0.24  
2        0.22  
3        0.16  
4        0.28  
<class 'pandas.core.frame.DataFrame'>
RangeI

## 2. Create Vocabularies
**MODIFIED:** This cell replaces the ESM loader. We now create a vocabulary for the input amino acid sequences (`seq_vocab`) in addition to the label vocabularies.

In [2]:
# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# NEW: Create vocabulary for input amino acid sequences
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0 # Add padding token
vocab_size = len(seq_vocab)

print(f"Sequence vocab size: {vocab_size}")
print(seq_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Sequence vocab size: 21
{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, '<pad>': 0}


## 3. Define Dataset and Collate Function
**MODIFIED:** The `ProteinDataset` now returns *tokenized sequences* instead of pre-computed embeddings. We also define a `collate_fn` to handle padding of sequences and labels at the batch level.

In [3]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        
        # Tokenize sequence
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq] # Default to <pad> if char not in vocab
        
        # Tokenize labels
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3]
        
        # Ensure label length matches sequence length
        ss8_tokens = ss8_tokens[:len(seq_tokens)]
        ss3_tokens = ss3_tokens[:len(seq_tokens)]
        
        return torch.tensor(seq_tokens, dtype=torch.long), torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    
    # Pad sequences
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    
    # Pad labels (use -1 for padding, as in original code)
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. Split Data and Create Dataloaders
**MODIFIED:** We split the dataframe indices and create the new `ProteinSequenceDataset`. The `DataLoader` now uses our custom `collate_fn`.

In [4]:
# Split indices (same as before)
train_indices, temp_indices = train_test_split(range(len(df)), test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

# Create datasets
train_dataset = ProteinSequenceDataset(
    df.iloc[train_indices]['seq'].tolist(),
    df.iloc[train_indices]['sst8'].tolist(),
    df.iloc[train_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
val_dataset = ProteinSequenceDataset(
    df.iloc[val_indices]['seq'].tolist(),
    df.iloc[val_indices]['sst8'].tolist(),
    df.iloc[val_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
test_dataset = ProteinSequenceDataset(
    df.iloc[test_indices]['seq'].tolist(),
    df.iloc[test_indices]['sst8'].tolist(),
    df.iloc[test_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)

# Create dataloaders with the collate_fn
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# NEW: Define embedding dim as a hyperparameter
embedding_dim = 128 

## 5. Define the Transformer Model
**MODIFIED:** This Transformer now includes its own `nn.Embedding` layer and a `PositionalEncoding` layer. It takes token IDs as input, not pre-computed embeddings.

In [5]:
class PositionalEncoding(nn.Module):
    """Standard Transformer Positional Encoding"""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class ProteinTransformer(nn.Module):
    def __init__(self, vocab_size, input_dim=128, num_heads=8, num_layers=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=seq_vocab['<pad>'])
        self.pos_encoder = PositionalEncoding(input_dim, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Two separate classifier heads
        self.q8_head = nn.Linear(input_dim, 8)
        self.q3_head = nn.Linear(input_dim, 3)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len] (token IDs)
        mask: [batch_size, seq_len] (padding mask, True where padded)
        """
        x = self.embedding(x) * math.sqrt(self.input_dim)
        x = self.pos_encoder(x)
        
        x = self.transformer_encoder(x, src_key_padding_mask=mask)
        
        # Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits


## 6. Training Loop
**MODIFIED:** The model is initialized with `vocab_size` and the new `embedding_dim`. The loop now iterates over `seqs, ss8, ss3` and creates the padding mask from the input `seqs`.

In [6]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# Initialize the model
model = ProteinTransformer(vocab_size=vocab_size, input_dim=embedding_dim)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

# Losses and optimizer
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 50
best_val_acc_q8 = 0.0

# --- Early Stopping Parameters ---
patience = 5  # Number of epochs to wait for improvement before stopping
counter = 0   # Counter for epochs without improvement
# ---------------------------------

for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc_q8, train_acc_q3 = 0, 0, 0

    for seqs, ss8, ss3 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)

        # Create padding mask from input sequences
        mask = (seqs == seq_vocab['<pad>'])

        # Forward pass
        q8_logits, q3_logits = model(seqs, mask)

        # Loss
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_acc_q8 += compute_accuracy(q8_logits, ss8)
        train_acc_q3 += compute_accuracy(q3_logits, ss3)

    train_loss /= len(train_loader)
    train_acc_q8 /= len(train_loader)
    train_acc_q3 /= len(train_loader)

    # Validation
    model.eval()
    val_loss, val_acc_q8, val_acc_q3 = 0, 0, 0
    with torch.no_grad():
        for seqs, ss8, ss3 in val_loader:
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            mask = (seqs == seq_vocab['<pad>'])
            q8_logits, q3_logits = model(seqs, mask)

            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3

            val_loss += loss.item()
            val_acc_q8 += compute_accuracy(q8_logits, ss8)
            val_acc_q3 += compute_accuracy(q3_logits, ss3)

    val_loss /= len(val_loader)
    val_acc_q8 /= len(val_loader)
    val_acc_q3 /= len(val_loader)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
    print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}")

    # --- Early Stopping & Checkpointing Logic ---
    if val_acc_q8 > best_val_acc_q8:
        best_val_acc_q8 = val_acc_q8
        counter = 0  # Reset counter since we found a better model
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(model_state, "best_scratch_transformer_model.pt")
        print("Validation accuracy improved. Model saved.")
    else:
        counter += 1
        print(f"EarlyStopping counter: {counter} out of {patience}")
        if counter >= patience:
            print("Early stopping triggered.")
            break
    # ---------------------------------------------

Epoch 1/50: 100%|██████████| 450/450 [00:09<00:00, 45.65it/s]
/home/users/ntu/ktang022/.conda/envs/myvenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1: Train Loss=2.0810, Val Loss=2.0396
Train Acc Q8=0.3768, Val Acc Q8=0.3930
Train Acc Q3=0.4956, Val Acc Q3=0.5173
Validation accuracy improved. Model saved.


Epoch 2/50: 100%|██████████| 450/450 [00:09<00:00, 49.34it/s]


Epoch 2: Train Loss=2.0379, Val Loss=2.0325
Train Acc Q8=0.3942, Val Acc Q8=0.3948
Train Acc Q3=0.5148, Val Acc Q3=0.5191
Validation accuracy improved. Model saved.


Epoch 3/50: 100%|██████████| 450/450 [00:09<00:00, 49.64it/s]


...

Epoch 14: Train Loss=2.0162, Val Loss=2.0248
Train Acc Q8=0.4029, Val Acc Q8=0.3974
Train Acc Q3=0.5219, Val Acc Q3=0.5193
EarlyStopping counter: 3 out of 5


Epoch 15/50: 100%|██████████| 450/450 [00:09<00:00, 49.21it/s]


Epoch 15: Train Loss=2.0162, Val Loss=2.0271
Train Acc Q8=0.4028, Val Acc Q8=0.3997
Train Acc Q3=0.5219, Val Acc Q3=0.5200
EarlyStopping counter: 4 out of 5


Epoch 16/50: 100%|██████████| 450/450 [00:09<00:00, 49.82it/s]


Epoch 16: Train Loss=2.0147, Val Loss=2.0276
Train Acc Q8=0.4035, Val Acc Q8=0.3971
Train Acc Q3=0.5225, Val Acc Q3=0.5170
EarlyStopping counter: 5 out of 5
Early stopping triggered.


## 7. Final Evaluation on Test Set
**MODIFIED:** Loads the new model and evaluates using the sequence-based `test_loader`.

In [7]:
# Initialize a new model instance
model = ProteinTransformer(vocab_size=vocab_size, input_dim=embedding_dim)
# Load the best model state
model.load_state_dict(torch.load("best_scratch_transformer_model.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0, 0, 0
with torch.no_grad():
    for seqs, ss8, ss3 in test_loader:
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        mask = (seqs == seq_vocab['<pad>'])
        q8_logits, q3_logits = model(seqs, mask)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")

Test Loss: 2.0178
Test Accuracy Q8: 0.4036
Test Accuracy Q3: 0.5227
